# Pipeline de Modelagem e Otimização de Hiperparâmetros

## Objetivos Principais
### Este notebook serve como um template padronizado e reutilizável para treinar, otimizar e comparar diferentes modelos de regressão multitarget. A partir do dataset processado, os objetivos são:

* #### **1. Treinamento Padronizado:** Aplicar um fluxo de trabalho consistente, carregando os dados limpos e utilizando a validação cruzada para avaliar a performance de diferentes algoritmos.
* #### **2. Otimização de Hiperparâmetros:** Utilizar `Optuna` para encontrar a melhor combinação de hiperparâmetros para cada modelo testado, maximizando sua performance preditiva.
* #### **3. Comparação de Desempenho:** Registrar e comparar de forma sistemática os scores da validação cruzada entre os diferentes modelos (ex: Ridge vs. RandomForest vs. LGBM) para identificar a abordagem mais eficaz.

### Fluxo Lógico:

- **1. Separar o dataset em features e targets;**

- **2. Dividir as features e targets em conjunto de treino e teste;**

- **3. Escalonar as features:**
O objetivo do escalonamento é colocar todas as features na mesma ordem de grandeza. Isso é feito para evitar que algoritmos sensíveis à escala deem uma importância desproporcional a uma feature simplesmente porque seus valores numéricos são maiores. Esta etapa é CRÍTICA para modelos **lineares** como o `Ridge`, que são sensíveis à escala; só é **aplicado quando necessário**;

- **4. Transformar o problema de regressão multitarget** em problemas simples de prever cada target separadamente. 
Esta é a abordagem de _"problem transformation"_, que neste caso (com `MultiOutputRegressor`) não leva em conta a correlação entre os **alvos (targets);** também serão testadas diferentes abordagens, como _"RegressorChain"_ e _"Algorithm adaptation"_

- **5. Definir o regressor base:** Neste exemplo, a Regressão Ridge. 
É uma variação da Regressão Linear que combate o overfitting através de uma penalidade. O hiperparâmetro alpha (que controla a força da penalidade) não é definido, pois vários valores serão testados afim de encontrar o melhor;

- **6. Otimizar e Avaliar**:
O `Optuna` realiza uma busca inteligente para encontrar os melhores hiperparâmetros, usando Validação Cruzada (`Cross Validation`) para garantir que a avaliação seja robusta e confiável;

- **7. Verificar os resultados e avaliar o modelo final:** 
Analisar os melhores hiperparâmetros encontrados pela busca e **verificar a performance do modelo otimizado** no conjunto de teste, que foi mantido separado durante todo o processo.

### Bibliotecas utilizadas:

In [7]:
import pandas as pd
import optuna
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from joblib import dump

### Leitura dos dados processados:

In [4]:
df = pd.read_csv('../data/meta/meta_processed/meta_proc_birds.csv')
df.head(3)

,normalize,feature preprocessing,mlfs.igmf_adapted.PyIT_IGMF,mlfs.igmf_adapted.PyIT_IGMF-n_features,mlfs.ppt_mi_adapted.PyIT_PPT_MI,mlfs.ppt_mi_adapted.PyIT_PPT_MI-n_features,mlfs.scls_adapted.PyIT_SCLS,mlfs.scls_adapted.PyIT_SCLS-n_features,mlfs.lrfs_adapted.PyIT_LRFS,mlfs.lrfs_adapted.PyIT_LRFS-n_features,...,weka.classifiers.functions.supportVector.NormalizedPolyKernel-E,weka.classifiers.functions.supportVector.NormalizedPolyKernel-L,weka.classifiers.functions.supportVector.Puk,weka.classifiers.functions.supportVector.Puk-O,weka.classifiers.functions.supportVector.Puk-S,weka.classifiers.functions.supportVector.RBFKernel,weka.classifiers.functions.supportVector.RBFKernel-G,F1 (macro averaged by label),Model Size,Model Size Log
0,0,1,0,-1.0,0,-1.0,0,-1.0,0,-1.0,...,-1.0,-1.0,0,-1.0,-1.0,0,-1.0,0.256,37076.0,10.520752
1,0,1,0,-1.0,0,-1.0,0,-1.0,0,-1.0,...,-1.0,-1.0,0,-1.0,-1.0,0,-1.0,0.222,29686.0,10.298465
2,0,1,0,-1.0,0,-1.0,0,-1.0,0,-1.0,...,-1.0,-1.0,0,-1.0,-1.0,0,-1.0,0.026,18387.0,9.819454


### **Dividindo o conjunto em treino e teste:**
#### Essa lógica não foi movida para nenhum arquivo na pasta `/src`; é usada da mesma forma nos experimentos.

In [ ]:
# definindo o caminho do dataset
data_path = f"../data/meta/meta_processed/meta_proc_birds.csv"

print("Lendo o Dataset...")
df = pd.read_csv(data_path)

print("\nDividindo dados em treino e teste...")

# Define os atributos preditivos
X = df.drop(columns=['F1 (macro averaged by label)', 'Model Size', 'Model Size Log'])

# Define os atributos alvo
y = df[['F1 (macro averaged by label)', 'Model Size Log']]

# Divide em treino e teste (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nDivisao concluida! Treinando o modelo: Ridge")

### **Treinando o modelo e otimizando os hiperparâmetros:**
#### Essa lógica foi movida para o arquivo `src/multi_output_training.py` (no caso da Ridge, mas outros modelos e abordagens também foram usados). 

In [ ]:

def objective(trial: optuna.Trial) -> float:
        # Pega uma sugestão de hiperparâmetros
        params = trial.suggest_float('alpha', 0.1, 100.0, log=True)
        
        # Constrói o pipeline 
        steps = []
        steps.append(StandardScaler()) # usa escalonamento
        steps.append(MultiOutputRegressor(Ridge(params))) 
        
        # Monta o modelo
        pipeline = make_pipeline(*steps)
        
        # Testa o modelo usando cross-validation de 3 folds
        scores = cross_val_score(pipeline, X_train, y_train, cv=3, scoring='r2')
        return scores.mean()

# Optuna otimiza os hiperparâmetros
print(f"\nIniciando Otimização: Ridge")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

# Último treino com os melhores parâmetros
print(f"Melhor R²: {study.best_value:.4f}")
final_steps = []
final_steps.append(StandardScaler())
final_steps.append(MultiOutputRegressor(Ridge(**study.best_params)))

best_model = make_pipeline(*final_steps)
best_model.fit(X_train, y_train)

### **Avaliação:**
#### Após obter o objeto do modelo treinado com os melhores hiperparâmetros, vamos calcular e exibir suas métricas de avaliação. Essa lógica foi movida para o arquivo `src/evaluate_model.py` e está disponível na função `evaluate_model_performance()`.

In [ ]:
# Obtém métricas de desempenho do modelo
print(f"\nAvaliação do Modelo: Ridge")
predictions = best_model.predict(X_test) # Faz inferência

# Extrai as colunas alvo do conjunto real
target_1_name = y_test.columns[0]
target_2_name = y_test.columns[1]

# Calcula R2 e RMSE para o primeiro alvo (F1 score)
r2_target_1 = r2_score(y_test.iloc[:, 0], predictions[:, 0])
rmse_target_1 = np.sqrt(mean_squared_error(y_test.iloc[:, 0], predictions[:, 0]))

# Calcula R2 e RMSE para o segundo alvo (tamanho do modelo em log)
r2_target_2 = r2_score(y_test.iloc[:, 1], predictions[:, 1])
rmse_target_2 = np.sqrt(mean_squared_error(y_test.iloc[:, 1], predictions[:, 1]))

# Monta o dicionário de resultados
metrics = {
    f"R2 para {target_1_name}": r2_target_1,
    f"R2 para {target_2_name}": r2_target_2,
    f"RMSE para {target_1_name}": rmse_target_1,
    f"RMSE para {target_2_name}": rmse_target_2
}

# Arredonda os valores para 4 casas decimais para melhor visualização
metrics = {key: round(value, 4) for key, value in metrics.items()}

# Exibe métricas
print("Resultados Finais no Conjunto de Teste:")
for metric_name, score in metrics.items():
    print(f"  - {metric_name}: {score}")

### **Salvando as métricas**:
#### Com as métricas calculadas, vamos salvá-las em um .CSV junto às métricas dos outros modelos que foram treinados no mesmo conjunto, para futura comparação. Essa lógica foi movida para o arquivo `src/evaluate_model.py` e está disponível na função `save_results()`

In [ ]:
# Salva os resultados da performance no CSV para comparação
print(f"\nSalvando Artefatos: Ridge")
path = f'../experiments_results/raw/raw_birds_results.csv'

# Cria um DataFrame com os novos resultados
results_df = pd.DataFrame({
    'Model': 'ridge_multi_output',
    'Best_Params': study.best_params,
    **metrics
})

# Garante que o diretório exista
output_dir = os.path.dirname(path)
os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(path):
    # Se o arquivo não existe, salva com o cabeçalho
    results_df.to_csv(path, index=False)
else:
    # Se o arquivo já existe, anexa sem o cabeçalho
    results_df.to_csv(path, mode='a', header=False, index=False)

print(f"Resultados para o modelo 'Ridge' salvos com sucesso em {path}")

### **Salvando o modelo treinado:**
#### Por último, vamos salvar o modelo (com joblib) para que possa ser reutilizado no futuro. Essa lógica foi movida para o arquivo `src/evaluate_model.py` e está disponível na função `save_model()`.

In [ ]:
model_path = f'../models/birds/ridge_multi_output.joblib'
print(f"Salvando o modelo em: {model_path}...")

# Garante que o diretório de destino exista
output_dir = os.path.dirname(model_path)
os.makedirs(output_dir, exist_ok=True)

# Salva o modelo no arquivo especificado
dump(best_model, model_path)

print("Modelo salvo com sucesso!")

### Aqui termina o processo de treinamento do modelo, cálculo das métricas e salvamento. Esse processo configura um único experimento, que será realizado em servidor. Ao todo serão realizados vários experimentos, variando o modelo, a abordagem e o conjunto de dados para decidir qual será o melhor candidato.